**Malzeme ve element analizi – kullanım defteri**

Eğitimi her seferinde baştan yapmamak için bu defteri hazırladım. Eğittiğim model Drive'da duruyor, burada sadece onu yüklüyorum. GPU gerekmiyor, 1-2 dakikada hazır oluyor.

Kullanım: Çalışma zamanı > Tümünü çalıştır. Drive izni isteyince onayla, sonra fotoğraf yükle.

### 1) Drive'daki modeli yükledim

In [ ]:
from google.colab import drive
import json, os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image

drive.mount('/content/drive')
klasor = '/content/drive/MyDrive/malzeme_projesi'

model = tf.keras.models.load_model(klasor + '/malzeme_modeli.keras')
sinif_adlari = json.load(open(klasor + '/siniflar.json'))
BOYUT = (224, 224)
print('model yüklendi, sınıflar:', sinif_adlari)

### 2) Element tabloları (eğitim defterindekiyle aynı)

In [ ]:
# yaklaşık kütlece yüzdeler
malzeme_bilesim = {
    'brick':         {'O': 48, 'Si': 28, 'Al': 9, 'Fe': 5, 'Ca': 3, 'K': 2, 'Mg': 1.5, 'Na': 1, 'Ti': 0.5, 'diğer': 2},
    'ceramic':       {'O': 50.1, 'Si': 32.7, 'Al': 13.2, 'K': 2.9, 'Ca': 1.1},
    'fabric':        {'O': 49.4, 'C': 44.4, 'H': 6.2},
    'foliage':       {'C': 45, 'O': 42, 'H': 6, 'N': 2.5, 'K': 1.5, 'Ca': 1.5, 'Mg': 0.3, 'P': 0.3, 'S': 0.3, 'diğer': 0.6},
    'glass':         {'O': 46.6, 'Si': 33.7, 'Na': 10.4, 'Ca': 6.4, 'Mg': 2.4, 'Al': 0.5},
    'leather':       {'C': 47, 'O': 26, 'N': 16, 'H': 6.5, 'Cr': 2.5, 'S': 0.5, 'diğer': 1.5},
    'metal':         {'Fe': 98, 'Mn': 0.8, 'Si': 0.3, 'C': 0.2, 'diğer': 0.7},
    'paper':         {'O': 49, 'C': 41, 'H': 5.8, 'Ca': 2.5, 'diğer': 1.7},
    'plastic':       {'C': 85.7, 'H': 14.3},
    'polishedstone': {'O': 48.5, 'Si': 33.5, 'Al': 7.5, 'K': 3.5, 'Na': 2.8, 'Fe': 2, 'Ca': 1.3, 'Mg': 0.4, 'Ti': 0.2, 'diğer': 0.3},
    'stone':         {'O': 48, 'Ca': 40, 'C': 12},
    'wood':          {'C': 50, 'O': 43, 'H': 6, 'N': 0.2, 'diğer': 0.8},
    # yedek veri seti (trashnet) için
    'cardboard':     {'O': 49.4, 'C': 44.4, 'H': 6.2},
    'trash':         {},
}

malzeme_tr = {
    'brick': 'tuğla', 'ceramic': 'seramik', 'fabric': 'kumaş (pamuk)', 'foliage': 'yaprak/bitki',
    'glass': 'cam', 'leather': 'deri', 'metal': 'metal (çelik)', 'paper': 'kağıt',
    'plastic': 'plastik (PE/PP)', 'polishedstone': 'cilalı taş (granit)', 'stone': 'taş (kireçtaşı)',
    'wood': 'ahşap', 'cardboard': 'karton', 'trash': 'karışık çöp',
}

In [ ]:
cam = malzeme_bilesim['glass']
seramik = malzeme_bilesim['ceramic']
pet = {'C': 62.5, 'O': 33.3, 'H': 4.2}
celik = {'Fe': 98.5, 'Mn': 0.8, 'Zn': 0.5, 'C': 0.2}  # galvanizli
pirinc = {'Cu': 65, 'Zn': 33, 'Pb': 1.5, 'diğer': 0.5}

# imagenet sınıf adı: (türkçe ad, bileşim)
nesne_bilesim = {
    'ballpoint':          ('tükenmez kalem', {'C': 80, 'H': 13, 'Cu': 3, 'Zn': 2, 'W': 1, 'diğer': 1}),
    'fountain_pen':       ('dolma kalem', {'C': 70, 'H': 11, 'Fe': 10, 'O': 5, 'Cr': 2.5, 'Ni': 1, 'diğer': 0.5}),
    'rubber_eraser':      ('silgi', {'C': 50, 'Cl': 35, 'H': 7, 'O': 5, 'Ca': 3}),
    'pencil_sharpener':   ('kalemtıraş', {'Mg': 90, 'Al': 6, 'Zn': 1, 'Fe': 3}),
    'beer_bottle':        ('cam şişe', cam),
    'wine_bottle':        ('şarap şişesi', cam),
    'beer_glass':         ('bardak', cam),
    'water_bottle':       ('pet şişe', pet),
    'pop_bottle':         ('gazoz şişesi (PET)', pet),
    'coffee_mug':         ('kupa', seramik),
    'teapot':             ('demlik', seramik),
    'plate':              ('tabak', seramik),
    'vase':               ('vazo', seramik),
    'frying_pan':         ('tava (teflon kaplı alüminyum)', {'Al': 92, 'F': 5.5, 'C': 1.5, 'Mg': 1}),
    'wok':                ('wok tava (karbon çeliği)', {'Fe': 99, 'C': 0.5, 'Mn': 0.5}),
    'chain':              ('zincir', celik),
    'nail':               ('çivi', celik),
    'screw':              ('vida', celik),
    'safety_pin':         ('çengelli iğne (paslanmaz çelik)', {'Fe': 72, 'Cr': 18, 'Ni': 8, 'Mn': 1.5, 'Si': 0.5}),
    'hammer':             ('çekiç', {'Fe': 75, 'C': 13, 'O': 10, 'H': 1.5, 'Mn': 0.5}),
    'padlock':            ('asma kilit (pirinç)', pirinc),
    'cornet':             ('kornet (pirinç)', {'Cu': 70, 'Zn': 30}),
    'trombone':           ('trombon (pirinç)', {'Cu': 70, 'Zn': 30}),
    'French_horn':        ('korno (pirinç)', {'Cu': 70, 'Zn': 30}),
    'necklace':           ('kolye (14 ayar altın varsaydım)', {'Au': 58.5, 'Cu': 30, 'Ag': 9, 'Zn': 2.5}),
    'cellular_telephone': ('cep telefonu', {'O': 22, 'C': 20, 'Al': 15, 'Si': 11, 'Cu': 10, 'Co': 5, 'Fe': 5, 'H': 3,
                                            'Ni': 1.5, 'Li': 0.7, 'Sn': 0.6, 'Mg': 0.5, 'Ag': 0.1, 'Au': 0.03, 'diğer': 5.57}),
    'laptop':             ('dizüstü bilgisayar', {'C': 25, 'O': 20, 'Al': 18, 'Si': 10, 'Cu': 8, 'Fe': 8, 'H': 3, 'Co': 3,
                                                  'Ni': 1, 'Mg': 1, 'Sn': 0.5, 'Li': 0.4, 'Ag': 0.05, 'Au': 0.01, 'diğer': 2.04}),
    'wooden_spoon':       ('tahta kaşık', malzeme_bilesim['wood']),
    'candle':             ('mum (parafin)', {'C': 85, 'H': 15}),
    'tennis_ball':        ('tenis topu', {'C': 70, 'H': 9, 'O': 8, 'Si': 5, 'Zn': 2, 'S': 2, 'N': 2, 'diğer': 2}),
    'golf_ball':          ('golf topu', {'C': 80, 'H': 12, 'O': 3, 'Zn': 3, 'diğer': 2}),
    'running_shoe':       ('spor ayakkabı', {'C': 70, 'O': 15, 'H': 10, 'N': 3, 'diğer': 2}),
    'carton':             ('karton kutu', malzeme_bilesim['cardboard']),
    'envelope':           ('zarf', malzeme_bilesim['paper']),
}

element_tr = {
    'H': 'hidrojen', 'Li': 'lityum', 'C': 'karbon', 'N': 'azot', 'O': 'oksijen', 'F': 'flor',
    'Na': 'sodyum', 'Mg': 'magnezyum', 'Al': 'alüminyum', 'Si': 'silisyum', 'P': 'fosfor',
    'S': 'kükürt', 'Cl': 'klor', 'K': 'potasyum', 'Ca': 'kalsiyum', 'Ti': 'titanyum',
    'Cr': 'krom', 'Mn': 'mangan', 'Fe': 'demir', 'Co': 'kobalt', 'Ni': 'nikel', 'Cu': 'bakır',
    'Zn': 'çinko', 'Ag': 'gümüş', 'Sn': 'kalay', 'W': 'tungsten', 'Au': 'altın', 'Pb': 'kurşun',
    'diğer': 'diğer',
}

# kontrol: toplamlar 100 mü, hepsinin türkçe adı var mı
tum_tablolar = list(malzeme_bilesim.items()) + [(k, v[1]) for k, v in nesne_bilesim.items()]
for ad, b in tum_tablolar:
    if b and abs(sum(b.values()) - 100) > 0.5:
        print('TOPLAM HATALI:', ad, sum(b.values()))
    for e in b:
        if e not in element_tr:
            print('türkçesi yok:', e)

farkli = sorted({e for _, b in tum_tablolar for e in b if e != 'diğer'})
print(len(farkli), 'farklı element:', ', '.join(farkli))

In [ ]:
# sonradan eklediğim nesneler
element_tr['V'] = 'vanadyum'
# tornavida: krom-vanadyum çelik uç (~%55) + plastik sap (~%45) varsaydım
nesne_bilesim['screwdriver'] = ('tornavida', {'Fe': 53.4, 'C': 38.9, 'H': 6.4, 'Cr': 0.55, 'Mn': 0.45, 'Si': 0.17, 'V': 0.11, 'diğer': 0.02})
nesne_bilesim['hand-held_computer'] = nesne_bilesim['cellular_telephone']
nesne_bilesim['iPod'] = nesne_bilesim['cellular_telephone']
nesne_bilesim['cup'] = ('fincan', seramik)

### 3) Nesne modeli ve tahmin fonksiyonları

In [ ]:
nesne_modeli = tf.keras.applications.MobileNetV2(weights='imagenet')  # 1000 sınıflı hazır model

def nesne_bul(img, kac=3):
    dizi = np.array(img.convert('RGB').resize(BOYUT), dtype='float32')[None, ...]
    dizi = tf.keras.applications.mobilenet_v2.preprocess_input(dizi)
    sonuc = tf.keras.applications.mobilenet_v2.decode_predictions(nesne_modeli.predict(dizi, verbose=0), top=kac)[0]
    return [(ad, float(p)) for _, ad, p in sonuc]

def malzeme_bul(img):
    dizi = np.array(img.convert('RGB').resize(BOYUT), dtype='float32')[None, ...]  # ön işleme modelin içinde
    olasilik = model.predict(dizi, verbose=0)[0]
    i = int(np.argmax(olasilik))
    return sinif_adlari[i], float(olasilik[i])

In [ ]:
def element_metni(bilesim):
    sirali = sorted(bilesim.items(), key=lambda t: -t[1])
    return ', '.join(f"%{v:g} {element_tr[k]} ({k})" if k != 'diğer' else f"%{v:g} diğer" for k, v in sirali)

def acikla(img, goster=True):
    nesneler = nesne_bul(img)
    malzeme, m_guven = malzeme_bul(img)

    secilen_nesne = None
    for ad, p in nesneler:
        if ad in nesne_bilesim and p > 0.2:
            secilen_nesne = (ad, p)
            break

    if secilen_nesne:
        ad, p = secilen_nesne
        tr_ad, bilesim = nesne_bilesim[ad]
        cumle = f"Bu büyük olasılıkla bir {tr_ad} (%{p*100:.0f}). "
        cumle += f"Malzeme modelim yüzeyi {malzeme_tr[malzeme]} olarak gördü (%{m_guven*100:.0f}). "
    else:
        bilesim = malzeme_bilesim[malzeme]
        if m_guven < 0.5:
            cumle = f"Nesneyi tanıyamadım, malzemeden de tam emin değilim. En yakın tahminim {malzeme_tr[malzeme]} (%{m_guven*100:.0f}). "
        else:
            cumle = f"Nesneyi tanıyamadım ama malzemesi büyük olasılıkla {malzeme_tr[malzeme]} (%{m_guven*100:.0f}). "

    if bilesim:
        cumle += f"İçinde {len([e for e in bilesim if e != 'diğer'])} farklı element var: {element_metni(bilesim)}."
    else:
        cumle += "Karışık olduğu için belli bir element oranı veremiyorum."
    cumle += " (Ölçüm değil, tipik bileşimdir.)"

    if goster:
        plt.imshow(img)
        plt.axis('off')
        plt.show()
        print('nesne modelinin ilk 3 tahmini:', [(a, round(p, 2)) for a, p in nesneler])
    print(cumle)
    return cumle

### 4) Fotoğraf yükleyip analiz ettim

In [ ]:
from google.colab import files
yuklenen = files.upload()
for ad in yuklenen:
    acikla(Image.open(ad))

### 5) Arayüz (isteğe bağlı)
Çalıştırınca bir link veriyor, telefondan da açılıyor.

In [ ]:
!pip install -q gradio
import gradio as gr

def arayuz(img):
    dizi = np.array(img.convert('RGB').resize(BOYUT), dtype='float32')[None, ...]
    olasilik = model.predict(dizi, verbose=0)[0]
    malzemeler = {malzeme_tr[s]: float(o) for s, o in zip(sinif_adlari, olasilik)}
    return malzemeler, acikla(img, goster=False)

gr.Interface(fn=arayuz, inputs=gr.Image(type='pil'),
             outputs=[gr.Label(num_top_classes=3, label='malzeme'), gr.Textbox(label='açıklama')],
             title='Nesne, malzeme ve element tahmini').launch(share=True)